# 3.25 — AdaBoost

AdaBoost turns many barely-better-than-guessing rules into one strong classifier by repeatedly asking, "which examples did the current ensemble embarrass itself on?" In this lesson, you will build the whole mechanism from scratch in NumPy: weighted errors, learner votes, exponential reweighting, margins, validation comparison, and the small cost/stability arithmetic from the lesson prose.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build AdaBoost one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is spelled out so the boosting loop is not a black box. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, weighted sums, and small numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any stochastic tie-breaking.

### 1. Start with labels in {-1, +1} and a weak rule

AdaBoost is a binary classifier built around labels $y_i\in\{-1,+1\}$. A weak learner is allowed to be simple — here, a one-threshold decision stump — as long as it performs slightly better than weighted guessing. We begin with equal example weights because, before seeing any mistakes, every training point deserves the same attention.

In [ ]:
x_w = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0])  # one feature, six ordered examples.
y_w = np.array([-1, -1, +1, +1, +1, -1])         # binary AdaBoost labels.
w_w = np.ones_like(x_w, dtype=float) / len(x_w)   # equal starting weights sum to 1.
print("x:", x_w)
print("y:", y_w)
print("initial weights:", np.round(w_w, 3), "sum=", round(float(w_w.sum()), 3))

▶ What you'll see: every example begins with weight 1/6, so the first stump is judged like ordinary error rate.

In [ ]:
pred_stump_w = np.where(x_w < 1.25, -1, +1)  # simple rule: left side is -1, right side is +1.
miss_w = pred_stump_w != y_w                 # mark exactly which examples this weak rule gets wrong.
print("stump predictions:", pred_stump_w)
print("mistakes:", miss_w.astype(int))
plt.figure(figsize=(5, 3))
plt.scatter(x_w[y_w == -1], y_w[y_w == -1], s=130 * w_w[y_w == -1], color="crimson", label="y=-1")
plt.scatter(x_w[y_w == +1], y_w[y_w == +1], s=130 * w_w[y_w == +1], color="teal", label="y=+1")
plt.axvline(1.25, color="black", linestyle="--", label="stump threshold")
plt.yticks([-1, 1]); plt.xlabel("x"); plt.ylabel("label")
plt.title("1: a weak decision stump"); plt.legend(); plt.show()

▶ What you'll see: the stump separates most points correctly, but the last negative point is on the wrong side.

*Why it's done this way: AdaBoost does not require a powerful learner; it requires a repeatable learner whose weighted error is below 0.5. The ensemble gets power by changing the weights and calling this weak learner again.*

### 2. Weighted error is the empirical risk AdaBoost cares about

The error for round $t$ is not "mistakes divided by $n$" after the first round. It is the total probability mass currently sitting on mistakes:

$$\varepsilon_t=\sum_i w_i\,\mathbf{1}\{h_t(x_i)\ne y_i\}.$$

This is the empirical-risk frame made concrete: the distribution over examples decides which mistakes are expensive. At round 1 the weights are equal, so weighted error equals ordinary error.

In [ ]:
eps_w = float(np.sum(w_w * miss_w))  # weighted error = total mass on wrong examples.
plain_err_w = float(np.mean(miss_w))  # ordinary error for comparison.
print("weighted error ε:", round(eps_w, 3))
print("ordinary error:", round(plain_err_w, 3))
assert round(eps_w, 3) == 0.167

▶ What you'll see: the rule misses 1 of 6 examples, so ε = 0.167 with equal weights.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(np.arange(len(x_w)), w_w, color=np.where(miss_w, "crimson", "gray"))
plt.title("2: weighted error is red mass")
plt.xlabel("example index"); plt.ylabel("weight")
plt.show()

▶ What you'll see: only the red bar contributes to ε; all correct examples contribute zero to this round's error.

*Why it's done this way: the weights are a movable empirical distribution. Summing mistake weights, not mistake counts, lets later rounds focus on the examples the ensemble has not yet explained.*

### 3. Convert error into a learner vote

A weak learner's vote is

$$\alpha_t=\tfrac12\ln\frac{1-\varepsilon_t}{\varepsilon_t}.$$

If $\varepsilon_t<0.5$, the log ratio is positive and the learner helps the ensemble. If $\varepsilon_t=0.5$, the vote is zero because the learner is no better than guessing. The factor $1/2$ is the exact step size that minimizes AdaBoost's exponential-loss bound for this fixed weak learner.

In [ ]:
alpha_w = 0.5 * np.log((1 - eps_w) / eps_w)  # AdaBoost vote for the weak learner.
print("alpha:", round(alpha_w, 3))
assert round(alpha_w, 3) == 0.805

▶ What you'll see: a low error of 0.167 earns a positive vote of about 0.805.

In [ ]:
eps_grid_w = np.linspace(0.01, 0.49, 80)
alpha_grid_w = 0.5 * np.log((1 - eps_grid_w) / eps_grid_w)
plt.figure(figsize=(5, 3))
plt.plot(eps_grid_w, alpha_grid_w, color="purple")
plt.scatter([eps_w], [alpha_w], color="red")
plt.axvline(0.5, color="black", linestyle="--")
plt.xlabel("weighted error ε"); plt.ylabel("vote α")
plt.title("3: better weak learners get larger votes")
plt.show()

▶ What you'll see: α rises sharply as ε gets small and shrinks toward 0 as ε approaches 0.5.

*Why it's done this way: the log-odds form compares correct mass to incorrect mass. A learner that is twice as reliable should not merely get a linear bonus; it gets an additive margin contribution calibrated by the log ratio.*

### 4. Exponential reweighting raises attention on mistakes

After choosing $\alpha_t$, AdaBoost updates each example weight by

$$w_i\leftarrow w_i e^{-\alpha_t y_i h_t(x_i)}.$$

The product $y_i h_t(x_i)$ is $+1$ when the prediction is correct and $-1$ when it is wrong. Therefore correct examples are multiplied by $e^{-\alpha_t}$, mistakes by $e^{+\alpha_t}$, and then everything is renormalized to sum to 1.

In [ ]:
margin_sign_w = y_w * pred_stump_w                       # +1 correct, -1 wrong.
raw_new_w = w_w * np.exp(-alpha_w * margin_sign_w)        # unnormalized AdaBoost update.
new_w = raw_new_w / raw_new_w.sum()                       # normalize into a distribution again.
print("multipliers:", np.round(np.exp(-alpha_w * margin_sign_w), 3))
print("new weights:", np.round(new_w, 3), "sum=", round(float(new_w.sum()), 3))
assert np.allclose(np.round(new_w, 3), [0.1, 0.1, 0.1, 0.1, 0.1, 0.5])

▶ What you'll see: the single mistake jumps to weight 0.5, while each correct example drops to 0.1.

In [ ]:
xpos_w = np.arange(len(x_w))
plt.figure(figsize=(5, 3))
plt.bar(xpos_w - 0.18, w_w, width=0.36, label="before", color="gray")
plt.bar(xpos_w + 0.18, new_w, width=0.36, label="after", color="teal")
plt.title("4: mistakes receive more mass")
plt.xlabel("example index"); plt.ylabel("weight"); plt.legend(); plt.show()

▶ What you'll see: the wrong point becomes half the next round's training distribution.

*Why it's done this way: the exponential rule is multiplicative, so it preserves positivity and converts margins directly into attention. Correct examples become cheaper; mistakes become more expensive; normalization keeps the next weighted error meaningful.*

### 5. The ensemble is a signed weighted sum of weak rules

AdaBoost does not average class labels directly. It builds a score

$$F_T(x)=\sum_{t=1}^T \alpha_t h_t(x),\qquad \hat y=\operatorname{sign}(F_T(x)).$$

The magnitude $|F_T(x)|$ is the ensemble's confidence-like margin size: a large positive score supports $+1$, a large negative score supports $-1$, and values near zero are fragile.

In [ ]:
h1_w = pred_stump_w
h2_w = np.where(x_w < 2.75, +1, -1)  # a second stump that specifically fixes the last negative point but hurts early negatives.
eps2_w = float(np.sum(new_w * (h2_w != y_w)))
alpha2_w = 0.5 * np.log((1 - eps2_w) / eps2_w)
print("round-2 weighted error:", round(eps2_w, 3), "alpha2:", round(alpha2_w, 3))
assert round(eps2_w, 3) == 0.2

▶ What you'll see: after reweighting, the second stump has weighted error 0.2 and earns a positive vote.

In [ ]:
F2_w = alpha_w * h1_w + alpha2_w * h2_w
pred2_w = np.sign(F2_w)
train_err2_w = float(np.mean(pred2_w != y_w))
print("ensemble scores:", np.round(F2_w, 3))
print("ensemble predictions:", pred2_w.astype(int), "training error:", round(train_err2_w, 3))
plt.figure(figsize=(5, 3))
plt.bar(np.arange(len(x_w)), F2_w, color=np.where(F2_w >= 0, "teal", "crimson"))
plt.axhline(0, color="black", linewidth=1)
plt.title("5: additive ensemble scores")
plt.xlabel("example index"); plt.ylabel("F(x)"); plt.show()

▶ What you'll see: the sign of each bar is the prediction, and bars near zero are the least secure decisions.

*Why it's done this way: adding weighted votes lets several crude rules correct one another. The sign makes the class decision, while the raw sum preserves margin information for understanding confidence and later loss calculations.*

### 6. Margins and exponential loss explain the pressure to improve

For a labeled example, the margin is $y_iF(x_i)$. Correct confident predictions have positive large margins; wrong predictions have negative margins. AdaBoost is commonly derived as greedily reducing exponential loss:

$$\sum_i e^{-y_iF(x_i)}.$$

Negative margins explode under this loss, so the next weak learner is pressured toward the stubborn mistakes.

In [ ]:
margins_w = y_w * F2_w
exp_losses_w = np.exp(-margins_w)
print("margins:", np.round(margins_w, 3))
print("exponential losses:", np.round(exp_losses_w, 3))

▶ What you'll see: correct high-margin examples have small loss, while weak or wrong margins remain expensive.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(np.arange(len(x_w)), exp_losses_w, color="darkorange")
plt.title("6: exponential loss emphasizes small margins")
plt.xlabel("example index"); plt.ylabel("exp(-yF(x))"); plt.show()

▶ What you'll see: examples with smaller margins produce taller bars, so they dominate the next improvement target.

*Why it's done this way: zero-one error only says right or wrong; exponential loss also cares how safely right. That margin pressure is why boosting can keep improving even after training accuracy looks good.*

### 7. Full selection uses raw fit, cost, gap, and stability

The lesson's content also keeps a broader model-selection habit visible. A training fragment alone is not the final decision. For the verified toy arithmetic, the raw empirical score averages three losses, then a cost is added, a more flexible alternative is compared, and a stabilizing knob is evaluated.

In [ ]:
losses_w = np.array([0.257, 0.070, 0.522])
raw_fit_w = float(losses_w.mean())
cost_w = 0.060
score_w = raw_fit_w + cost_w
print("raw average R_S:", round(raw_fit_w, 3))
print("score with cost:", round(score_w, 3))
assert round(raw_fit_w, 3) == 0.283 and round(score_w, 3) == 0.343

▶ What you'll see: the raw average is 0.283, but the selection score is 0.343 after adding the cost guardrail.

In [ ]:
flexible_w = 0.379
gap_w = flexible_w - score_w
relative_gap_w = gap_w / flexible_w
stable_w = 0.80 * score_w
candidates_w = np.array([score_w, flexible_w, stable_w])
print("gap:", round(gap_w, 3), "relative gap:", round(relative_gap_w, 3))
print("stable score:", round(stable_w, 3), "winner:", round(float(candidates_w.min()), 3))
assert round(gap_w, 3) == 0.036 and round(relative_gap_w, 3) == 0.095 and round(stable_w, 3) == 0.274
plt.figure(figsize=(5, 3))
plt.bar(["baseline", "flexible", "stable"], candidates_w, color=["gray", "crimson", "teal"])
plt.ylabel("decision score (lower is better)"); plt.title("7: compare full scores, not fragments"); plt.show()

▶ What you'll see: the stabilized score is the lowest of the three verified candidates.

*Why it's done this way: AdaBoost's formula tells us how to optimize the ensemble, but validation and cost tell us whether that optimized flexibility is trustworthy. The decision unit is the full score, not the prettiest raw training number.*


## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per mechanic in this lesson. Each uses a handful of small
> numbers, prints every intermediate value with an inline `# ->` showing the result, draws one
> picture, and ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · Signed stump predictions make a mistake mask

AdaBoost uses labels and weak-rule outputs in `{-1,+1}` so correctness can become arithmetic. A
single threshold stump below gets one row wrong.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)
t1_x = np.array([0, 1, 2, 3, 4, 5])
t1_y = np.array([-1, -1, -1, 1, 1, -1])
t1_threshold = 2.5
t1_pred = np.where(t1_x < t1_threshold, -1, 1)          # -> [-1 -1 -1  1  1  1]
print("x:", t1_x.tolist())                              # -> [0, 1, 2, 3, 4, 5]
print("signed labels:", t1_y.tolist())                  # -> [-1, -1, -1, 1, 1, -1]
print("stump predictions:", t1_pred.tolist())           # -> [-1, -1, -1, 1, 1, 1]
t1_miss = t1_pred != t1_y                                # -> [False False False False False  True]
print("mistake mask:", t1_miss.astype(int).tolist())     # -> [0, 0, 0, 0, 0, 1]
t1_mistakes = int(t1_miss.sum())                         # -> 1
print("mistake count:", t1_mistakes)                     # -> 1
assert t1_mistakes == 1

plt.figure(figsize=(4.8, 2.8))
plt.scatter(t1_x, t1_y, s=90, label="true y")
plt.scatter(t1_x, t1_pred, marker="x", s=90, label="stump h")
plt.axvline(t1_threshold, color="black", linestyle="--")
plt.yticks([-1, 1])
plt.xlabel("x")
plt.ylabel("signed class")
plt.title("Toy 1 · one threshold mistake")
plt.legend()
plt.show()

▶ What you'll see: the threshold gets only the last negative example wrong.

### ✍️ Toy 2 · Weighted error sums the wrong mass

Once weights are not uniform, AdaBoost scores a weak learner by the probability mass sitting on its
mistakes, not by a plain count.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)
t2_weights = np.array([0.10, 0.10, 0.15, 0.15, 0.20, 0.30])
t2_miss = np.array([False, True, False, False, True, False])
t2_parts = t2_weights * t2_miss                          # -> [0.   0.1  0.   0.   0.2  0.  ]
print("weights:", np.round(t2_weights, 2).tolist())       # -> [0.1, 0.1, 0.15, 0.15, 0.2, 0.3]
print("mistakes:", t2_miss.astype(int).tolist())          # -> [0, 1, 0, 0, 1, 0]
print("weighted parts:", np.round(t2_parts, 2).tolist())  # -> [0.0, 0.1, 0.0, 0.0, 0.2, 0.0]
t2_eps = float(t2_parts.sum())                            # -> 0.3
print("weighted error epsilon:", round(t2_eps, 3))        # -> 0.3
t2_alpha = 0.5 * np.log((1 - t2_eps) / t2_eps)            # -> 0.42364893019360184
print("alpha vote:", round(float(t2_alpha), 3))           # -> 0.424
assert round(t2_eps, 3) == 0.300 and round(float(t2_alpha), 3) == 0.424

plt.figure(figsize=(4.8, 2.8))
plt.bar(np.arange(6), t2_parts, color=np.where(t2_miss, "crimson", "gray"))
plt.xlabel("example")
plt.ylabel("weight × wrong")
plt.title("Toy 2 · epsilon is red mass")
plt.show()

▶ What you'll see: the two wrong rows contribute `0.10 + 0.20 = 0.30` to epsilon.

### ✍️ Toy 3 · Exponential reweighting focuses on mistakes

Correct examples are multiplied by `exp(-alpha)` and wrong examples by `exp(+alpha)`, then all weights
are normalized back to a distribution.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)
t3_weights = np.ones(6) / 6
t3_y = np.array([-1, -1, -1, 1, 1, -1])
t3_pred = np.array([-1, -1, -1, 1, 1, 1])
t3_sign = t3_y * t3_pred                                  # -> [ 1  1  1  1  1 -1]
print("old weights:", np.round(t3_weights, 3).tolist())   # -> [0.167, 0.167, 0.167, 0.167, 0.167, 0.167]
print("y*h signs:", t3_sign.tolist())                     # -> [1, 1, 1, 1, 1, -1]
t3_eps = float(np.sum(t3_weights * (t3_sign < 0)))         # -> 0.16666666666666666
t3_alpha = 0.5 * np.log((1 - t3_eps) / t3_eps)             # -> 0.8047189562170501
print("epsilon:", round(t3_eps, 3))                       # -> 0.167
print("alpha:", round(float(t3_alpha), 3))                # -> 0.805
t3_mult = np.exp(-t3_alpha * t3_sign)                      # -> [0.447 0.447 0.447 0.447 0.447 2.236]
print("multipliers:", np.round(t3_mult, 3).tolist())      # -> [0.447, 0.447, 0.447, 0.447, 0.447, 2.236]
t3_raw = t3_weights * t3_mult                              # -> [0.075 0.075 0.075 0.075 0.075 0.373]
print("raw new weights:", np.round(t3_raw, 3).tolist())   # -> [0.075, 0.075, 0.075, 0.075, 0.075, 0.373]
t3_new = t3_raw / t3_raw.sum()                             # -> [0.1 0.1 0.1 0.1 0.1 0.5]
print("normalized weights:", np.round(t3_new, 3).tolist()) # -> [0.1, 0.1, 0.1, 0.1, 0.1, 0.5]
assert np.allclose(np.round(t3_new, 3), [0.1, 0.1, 0.1, 0.1, 0.1, 0.5])

plt.figure(figsize=(4.8, 2.8))
plt.bar(np.arange(6) - 0.18, t3_weights, width=0.36, color="gray", label="before")
plt.bar(np.arange(6) + 0.18, t3_new, width=0.36, color="teal", label="after")
plt.xlabel("example")
plt.ylabel("weight")
plt.title("Toy 3 · the miss gets half the mass")
plt.legend()
plt.show()

▶ What you'll see: the single mistake jumps from weight `0.167` to `0.500`.

### ✍️ Toy 4 · Add votes, then read margins and exponential loss

AdaBoost accumulates raw scores before taking a sign. Multiplying by the true label turns those scores
into margins, and `exp(-margin)` spotlights fragile points.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)
t4_y = np.array([-1, -1, -1, 1, 1, -1])
t4_h1 = np.array([-1, -1, -1, 1, 1, 1])
t4_h2 = np.array([-1, -1, -1, 1, 1, -1])
t4_alpha1 = 0.6
t4_alpha2 = 0.8
print("alpha1:", t4_alpha1)                               # -> 0.6
print("alpha2:", t4_alpha2)                               # -> 0.8
t4_score = t4_alpha1 * t4_h1 + t4_alpha2 * t4_h2           # -> [-1.4 -1.4 -1.4  1.4  1.4 -0.2]
print("ensemble score:", np.round(t4_score, 3).tolist())   # -> [-1.4, -1.4, -1.4, 1.4, 1.4, -0.2]
t4_pred = np.sign(t4_score)                                # -> [-1. -1. -1.  1.  1. -1.]
print("predictions:", t4_pred.astype(int).tolist())        # -> [-1, -1, -1, 1, 1, -1]
t4_margin = t4_y * t4_score                                # -> [1.4 1.4 1.4 1.4 1.4 0.2]
print("margins:", np.round(t4_margin, 3).tolist())         # -> [1.4, 1.4, 1.4, 1.4, 1.4, 0.2]
t4_exp_loss = np.exp(-t4_margin)                           # -> [0.247 0.247 0.247 0.247 0.247 0.819]
print("exp losses:", np.round(t4_exp_loss, 3).tolist())    # -> [0.247, 0.247, 0.247, 0.247, 0.247, 0.819]
assert np.all(t4_margin > 0) and round(float(t4_exp_loss.max()), 3) == 0.819

plt.figure(figsize=(4.8, 2.8))
plt.bar(np.arange(6), t4_margin, color=np.where(t4_margin < 0.5, "orange", "teal"))
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("example")
plt.ylabel("margin yF(x)")
plt.title("Toy 4 · small positive margins stay costly")
plt.show()

▶ What you'll see: all examples are correct, but the two small-margin rows have the largest exponential loss.

### ✍️ Toy 5 · Search thresholds and polarities for the best stump

A weak-learner call is a small optimization problem: try candidate thresholds and both orientations,
then keep the rule with the smallest weighted error.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)
t5_x = np.array([0, 1, 2, 3, 4, 5])
t5_y = np.array([-1, -1, 1, 1, 1, -1])
t5_w = np.array([0.05, 0.10, 0.10, 0.20, 0.25, 0.30])
t5_thresholds = np.array([0.5, 1.5, 2.5, 3.5, 4.5])
print("thresholds:", t5_thresholds.tolist())               # -> [0.5, 1.5, 2.5, 3.5, 4.5]
print("weights:", np.round(t5_w, 2).tolist())              # -> [0.05, 0.1, 0.1, 0.2, 0.25, 0.3]
t5_errors = []
t5_rules = []
for t5_left_label in [-1, 1]:
    t5_row = []
    for t5_thr in t5_thresholds:
        t5_pred = np.where(t5_x < t5_thr, t5_left_label, -t5_left_label)
        t5_err = float(np.sum(t5_w * (t5_pred != t5_y)))
        t5_row.append(t5_err)
        t5_rules.append((t5_err, t5_thr, t5_left_label, t5_pred.copy()))
    t5_errors.append(t5_row)
t5_errors = np.array(t5_errors)                            # -> [[0.4  0.3  0.4  0.6  0.85] [0.6  0.7  0.6  0.4  0.15]]
print("error grid rows left=-1,+1:", np.round(t5_errors, 3).tolist()) # -> [[0.4, 0.3, 0.4, 0.6, 0.85], [0.6, 0.7, 0.6, 0.4, 0.15]]
t5_best = min(t5_rules, key=lambda t5_item: t5_item[0])
t5_best_err = float(t5_best[0])                            # -> 0.15000000000000002
t5_best_thr = float(t5_best[1])                            # -> 4.5
t5_best_left = int(t5_best[2])                             # -> 1
t5_best_pred = t5_best[3]                                  # -> [ 1  1  1  1  1 -1]
print("best error:", round(t5_best_err, 3))                # -> 0.15
print("best threshold and left label:", t5_best_thr, t5_best_left) # -> 4.5 1
print("best predictions:", t5_best_pred.tolist())          # -> [1, 1, 1, 1, 1, -1]
assert round(t5_best_err, 3) == 0.150 and t5_best_thr == 4.5 and t5_best_left == 1

plt.figure(figsize=(4.8, 2.8))
plt.plot(t5_thresholds, t5_errors[0], marker="o", label="left=-1")
plt.plot(t5_thresholds, t5_errors[1], marker="s", label="left=+1")
plt.axvline(t5_best_thr, color="red", linestyle="--")
plt.xlabel("threshold")
plt.ylabel("weighted error")
plt.title("Toy 5 · best weighted stump")
plt.legend()
plt.show()

▶ What you'll see: the best rule uses threshold `4.5` with the left side labeled `+1`.

### ✍️ Toy 6 · Validation can choose a shorter boosted ensemble

More rounds can keep improving training predictions while validation prefers an earlier ensemble size.
Here the second round is best on validation even though the third fixes training.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)
t6_train_y = np.array([-1, -1, 1, 1, 1, -1])
t6_val_y = np.array([-1, 1, 1, -1])
t6_alphas = np.array([0.6, 0.7, 0.4])
t6_h_train = np.array([[-1, -1, -1, 1, 1, -1], [-1, -1, 1, 1, 1, 1], [-1, -1, 1, 1, 1, -1]])
t6_h_val = np.array([[-1, 1, -1, -1], [-1, 1, 1, -1], [1, -1, -1, 1]])
print("alphas:", t6_alphas.tolist())                       # -> [0.6, 0.7, 0.4]
t6_train_err = []
t6_val_err = []
for t6_rounds in [1, 2, 3]:
    t6_train_score = np.sum(t6_alphas[:t6_rounds, None] * t6_h_train[:t6_rounds], axis=0)
    t6_val_score = np.sum(t6_alphas[:t6_rounds, None] * t6_h_val[:t6_rounds], axis=0)
    t6_train_pred = np.sign(t6_train_score)
    t6_val_pred = np.sign(t6_val_score)
    t6_train_err.append(float(np.mean(t6_train_pred != t6_train_y)))
    t6_val_err.append(float(np.mean(t6_val_pred != t6_val_y)))
print("train errors:", np.round(t6_train_err, 3).tolist())  # -> [0.167, 0.167, 0.0]
print("validation errors:", np.round(t6_val_err, 3).tolist()) # -> [0.25, 0.0, 0.25]
t6_best_round = int(np.argmin(t6_val_err)) + 1              # -> 2
print("best validation round:", t6_best_round)             # -> 2
assert t6_best_round == 2 and round(t6_train_err[-1], 3) == 0.0

plt.figure(figsize=(4.8, 2.8))
plt.plot([1, 2, 3], t6_train_err, marker="o", label="train")
plt.plot([1, 2, 3], t6_val_err, marker="s", label="validation")
plt.axvline(t6_best_round, color="red", linestyle="--")
plt.xlabel("boosting rounds")
plt.ylabel("classification error")
plt.title("Toy 6 · validation picks round 2")
plt.legend()
plt.show()

▶ What you'll see: training error is lowest at round 3, but validation is best at round 2.

### ✍️ Toy 7 · Full score adds cost, gap, and stabilization

The model-selection arithmetic is separate from AdaBoost's training loop: average losses, add cost,
compare a flexible alternative, then apply the stabilizing candidate.

In [ ]:
import numpy as np

t7_rng = np.random.default_rng(0)
t7_losses = np.array([0.18, 0.22, 0.16, 0.20, 0.24, 0.26])
t7_cost = 0.05
t7_flexible = 0.31
print("losses:", np.round(t7_losses, 2).tolist())          # -> [0.18, 0.22, 0.16, 0.2, 0.24, 0.26]
t7_raw = float(t7_losses.mean())                           # -> 0.21
print("raw average:", round(t7_raw, 3))                    # -> 0.21
t7_score = t7_raw + t7_cost                                 # -> 0.26
print("score with cost:", round(t7_score, 3))              # -> 0.26
t7_gap = t7_flexible - t7_score                             # -> 0.04999999999999999
t7_relative_gap = t7_gap / t7_flexible                      # -> 0.16129032258064513
t7_stable = 0.80 * t7_score                                 # -> 0.20800000000000002
print("gap:", round(t7_gap, 3))                            # -> 0.05
print("relative gap:", round(t7_relative_gap, 3))          # -> 0.161
print("stable score:", round(t7_stable, 3))                # -> 0.208
t7_scores = np.array([t7_score, t7_flexible, t7_stable])
t7_names = np.array(["baseline", "flexible", "stable"])
t7_winner = t7_names[int(np.argmin(t7_scores))]             # -> stable
print("winner:", t7_winner)                                # -> stable
assert t7_winner == "stable" and round(t7_stable, 3) == 0.208

plt.figure(figsize=(4.8, 2.8))
plt.bar(t7_names, t7_scores, color=["gray", "crimson", "teal"])
plt.ylabel("decision score")
plt.title("Toy 7 · lower full score wins")
plt.show()

▶ What you'll see: the stabilized score is the lowest after all candidates are put on the same scale.

### ✍️ Toy 8 · A noisy point can attract attention

A mislabeled outlier can repeatedly become expensive. This tiny loop trains stumps and records the
weight of the flipped row after each round.

In [ ]:
import numpy as np

t8_rng = np.random.default_rng(0)
t8_x = np.arange(7, dtype=float)
t8_y_clean = np.where(t8_x < 3, -1, 1)
t8_y = t8_y_clean.copy()
t8_y[1] = 1
t8_thresholds = (t8_x[:-1] + t8_x[1:]) / 2
t8_w = np.ones_like(t8_x) / len(t8_x)
t8_outlier = 1
t8_outlier_weights = [float(t8_w[t8_outlier])]
t8_train_errors = []
t8_score = np.zeros_like(t8_x, dtype=float)
print("noisy labels:", t8_y.astype(int).tolist())          # -> [-1, 1, -1, 1, 1, 1, 1]
print("initial outlier weight:", round(t8_outlier_weights[0], 3)) # -> 0.143
for t8_round in range(4):
    t8_best = None
    for t8_thr in t8_thresholds:
        for t8_left in [-1, 1]:
            t8_pred = np.where(t8_x < t8_thr, t8_left, -t8_left)
            t8_err = float(np.sum(t8_w * (t8_pred != t8_y)))
            if t8_err > 0.5:
                t8_err = 1 - t8_err
                t8_pred = -t8_pred
            if t8_best is None or t8_err < t8_best[0]:
                t8_best = (t8_err, t8_thr, t8_pred.copy())
    t8_err, t8_thr, t8_pred = t8_best
    t8_alpha = 0.5 * np.log((1 - t8_err) / max(t8_err, 1e-12))
    t8_score = t8_score + t8_alpha * t8_pred
    t8_w = t8_w * np.exp(-t8_alpha * t8_y * t8_pred)
    t8_w = t8_w / t8_w.sum()
    t8_outlier_weights.append(float(t8_w[t8_outlier]))
    t8_train_errors.append(float(np.mean(np.sign(t8_score) != t8_y)))
print("outlier weights:", np.round(t8_outlier_weights, 3).tolist()) # -> [0.143, 0.083, 0.5, 0.324, 0.196]
print("train errors:", np.round(t8_train_errors, 3).tolist()) # -> [0.143, 0.143, 0.0, 0.0]
print("final weights:", np.round(t8_w, 3).tolist())        # -> [0.061, 0.196, 0.5, 0.061, 0.061, 0.061, 0.061]
assert max(t8_outlier_weights) > t8_outlier_weights[0]

plt.figure(figsize=(4.8, 2.8))
plt.plot(range(len(t8_outlier_weights)), t8_outlier_weights, marker="o", color="crimson")
plt.xlabel("round")
plt.ylabel("weight on flipped row")
plt.title("Toy 8 · noisy example gets attention")
plt.show()

▶ What you'll see: the flipped row's weight spikes above its starting value during boosting.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, weighted errors, margins, and small numerical checks.
import matplotlib.pyplot as plt # load Matplotlib so every boosted step can be inspected visually.
np.random.seed(0) # make all examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Encode labels as -1 and +1

**Goal.** Build the tiny binary dataset AdaBoost expects, because the weight update uses the product y·h and needs labels in {-1,+1}. We build it in 2 steps.

In [ ]:
x_b1 = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0]) # store one feature for six ordered training examples.
y_b1 = np.array([-1, -1, 1, 1, 1, -1]) # store binary labels in AdaBoost's -1/+1 convention.
print("x_b1:", x_b1) # inspect the feature values.
print("y_b1:", y_b1) # inspect the signed class labels.

▶ What you'll see: a one-dimensional training set with a late negative point that will be hard for one threshold.

In [ ]:
plt.figure(figsize=(5, 3)) # create a compact label plot.
plt.scatter(x_b1, y_b1, c=np.where(y_b1 > 0, "teal", "crimson"), s=80) # draw each point at its signed class value.
plt.yticks([-1, 1]) # show only the two valid AdaBoost label values.
plt.title("Basic 1: signed labels") # title the figure.
plt.xlabel("x") # label the feature axis.
plt.ylabel("y") # label the signed class axis.
plt.show() # display the plot.

▶ What you'll see: positive and negative examples are not perfectly separable by a single threshold.

👀 Takeaway: AdaBoost's compact math depends on signed labels, where correct predictions make y·h = +1.

### Basic 2 — Initialize example weights

**Goal.** Give every example equal probability mass, because the first weak learner has no reason to favor one point over another. We build it in 2 steps.

In [ ]:
n_b2 = 6 # define the number of training examples.
w_b2 = np.ones(n_b2) / n_b2 # create a uniform distribution over examples.
print("weights:", np.round(w_b2, 3)) # inspect each example's starting mass.
print("sum:", round(float(w_b2.sum()), 3)) # verify the weights form a probability distribution.
assert round(float(w_b2.sum()), 3) == 1.0 # self-check that the distribution is normalized.

▶ What you'll see: all six weights are 0.167 and sum to exactly 1.

In [ ]:
plt.figure(figsize=(5, 3)) # create a compact weight chart.
plt.bar(np.arange(n_b2), w_b2, color="gray") # draw one bar per example weight.
plt.title("Basic 2: uniform starting weights") # title the chart.
plt.xlabel("example index") # label the example axis.
plt.ylabel("weight") # label the weight axis.
plt.show() # display the weights.

▶ What you'll see: every example has the same height before the first weak learner is fit.

👀 Takeaway: AdaBoost begins as ordinary empirical risk, then changes the distribution after mistakes appear.

### Basic 3 — Make stump predictions

**Goal.** Implement one decision stump by hand, because AdaBoost only needs a weak rule returning -1 or +1. We build it in 2 steps.

In [ ]:
x_b3 = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0]) # recreate the feature values for this example.
threshold_b3 = 1.25 # choose a threshold between 1.0 and 1.5.
pred_b3 = np.where(x_b3 < threshold_b3, -1, 1) # predict -1 on the left and +1 on the right.
print("threshold:", threshold_b3) # inspect the split point.
print("predictions:", pred_b3) # inspect the stump output.

▶ What you'll see: the first two points are predicted -1 and all later points are predicted +1.

In [ ]:
plt.figure(figsize=(5, 3)) # create a compact threshold plot.
plt.scatter(x_b3, pred_b3, c=np.where(pred_b3 > 0, "teal", "crimson"), s=80) # visualize predicted classes.
plt.axvline(threshold_b3, color="black", linestyle="--") # mark the stump split.
plt.yticks([-1, 1]) # show signed prediction values.
plt.title("Basic 3: stump predictions") # title the plot.
plt.xlabel("x") # label the feature axis.
plt.ylabel("h(x)") # label the weak learner output.
plt.show() # display the plot.

▶ What you'll see: a piecewise-constant classifier with only one split.

👀 Takeaway: a weak learner can be extremely simple as long as it beats weighted random guessing.

### Basic 4 — Mark mistakes

**Goal.** Compare stump predictions with true labels, because AdaBoost's next two computations need the mistake mask. We build it in 2 steps.

In [ ]:
y_b4 = np.array([-1, -1, 1, 1, 1, -1]) # define the true signed labels.
pred_b4 = np.array([-1, -1, 1, 1, 1, 1]) # define the stump predictions from Basic 3.
miss_b4 = pred_b4 != y_b4 # mark wrong predictions as True.
print("mistake mask:", miss_b4.astype(int)) # inspect 1 for wrong and 0 for correct.

▶ What you'll see: only the last example is misclassified by this stump.

In [ ]:
correct_b4 = pred_b4 == y_b4 # mark correct predictions for the companion visualization.
print("correct count:", int(np.sum(correct_b4)), "mistake count:", int(np.sum(miss_b4))) # inspect the raw counts.
plt.figure(figsize=(5, 3)) # create a compact correctness chart.
plt.bar(np.arange(len(y_b4)), miss_b4.astype(int), color=np.where(miss_b4, "crimson", "gray")) # highlight the mistake.
plt.title("Basic 4: one stump mistake") # title the chart.
plt.xlabel("example index") # label the example axis.
plt.ylabel("1 if wrong") # label the mistake indicator.
plt.show() # display the chart.

▶ What you'll see: one red bar at the final example.

👀 Takeaway: the mistake mask decides which weights enter the round's weighted error.

### Basic 5 — Compute weighted error

**Goal.** Sum the weights on wrong examples, because AdaBoost evaluates weak learners under the current example distribution. We build it in 2 steps.

In [ ]:
w_b5 = np.ones(6) / 6 # start with equal example weights.
miss_b5 = np.array([False, False, False, False, False, True]) # reuse the one-mistake stump.
weighted_parts_b5 = w_b5 * miss_b5 # keep weight only where the stump is wrong.
print("weighted mistake parts:", np.round(weighted_parts_b5, 3)) # inspect contributions to epsilon.

▶ What you'll see: five zeros and one 0.167 contribution from the wrong point.

In [ ]:
eps_b5 = float(np.sum(weighted_parts_b5)) # compute weighted error epsilon.
print("epsilon:", round(eps_b5, 3)) # inspect the weighted error.
assert round(eps_b5, 3) == 0.167 # verify the canonical first-round error.
plt.figure(figsize=(5, 3)) # create a compact error-mass plot.
plt.bar(np.arange(6), weighted_parts_b5, color="crimson") # draw only mistake mass.
plt.title("Basic 5: weighted error mass") # title the plot.
plt.xlabel("example index") # label the example axis.
plt.ylabel("weight × wrong") # label the contribution axis.
plt.show() # display the plot.

▶ What you'll see: weighted error is the total height of the red mistake-mass bars.

👀 Takeaway: ε is an error rate only when weights are uniform; later it is a focused cost.

### Basic 6 — Convert error to alpha

**Goal.** Turn weighted error into a learner vote, because stronger weak learners should move the ensemble more. We build it in 2 steps.

In [ ]:
eps_b6 = 1 / 6 # use the first-round weighted error from the one-mistake stump.
alpha_b6 = 0.5 * np.log((1 - eps_b6) / eps_b6) # compute AdaBoost's vote strength.
print("alpha:", round(alpha_b6, 3)) # inspect the vote.
assert round(alpha_b6, 3) == 0.805 # verify the canonical alpha value.

▶ What you'll see: one error out of six gives the stump a vote of about 0.805.

In [ ]:
eps_values_b6 = np.array([0.1, 0.2, 0.3, 0.4, 0.49]) # choose several possible weighted errors.
alpha_values_b6 = 0.5 * np.log((1 - eps_values_b6) / eps_values_b6) # compute votes for each error.
print("alpha grid:", np.round(alpha_values_b6, 3)) # inspect how votes shrink as error rises.
plt.figure(figsize=(5, 3)) # create a compact vote curve.
plt.plot(eps_values_b6, alpha_values_b6, marker="o", color="purple") # draw alpha against epsilon.
plt.title("Basic 6: alpha from weighted error") # title the chart.
plt.xlabel("ε") # label weighted error.
plt.ylabel("α") # label vote strength.
plt.show() # display the chart.

▶ What you'll see: alpha approaches zero as the weak learner approaches random guessing.

👀 Takeaway: AdaBoost rewards weak learners according to their weighted log-odds of correctness.

### Basic 7 — Compute correctness signs y·h

**Goal.** Convert correctness into +1 or -1 algebra, because the update formula uses y_i h_t(x_i) rather than an if-statement. We build it in 2 steps.

In [ ]:
y_b7 = np.array([-1, -1, 1, 1, 1, -1]) # define true labels.
h_b7 = np.array([-1, -1, 1, 1, 1, 1]) # define weak predictions.
signs_b7 = y_b7 * h_b7 # multiply labels by predictions to get +1 correct and -1 wrong.
print("y*h signs:", signs_b7) # inspect the algebraic correctness code.

▶ What you'll see: five +1 values and one -1 value.

In [ ]:
plt.figure(figsize=(5, 3)) # create a compact sign plot.
plt.bar(np.arange(len(signs_b7)), signs_b7, color=np.where(signs_b7 > 0, "teal", "crimson")) # show correct and wrong signs.
plt.axhline(0, color="black", linewidth=1) # separate correct from wrong.
plt.title("Basic 7: correctness as y·h") # title the chart.
plt.xlabel("example index") # label examples.
plt.ylabel("y_i h(x_i)") # label the signed correctness value.
plt.show() # display the chart.

▶ What you'll see: the wrong example is the only bar below zero.

👀 Takeaway: the product y·h lets one exponential formula shrink correct weights and grow mistaken weights.

### Basic 8 — Update weights once

**Goal.** Apply the exponential weight update, because AdaBoost focuses the next weak learner on current mistakes. We build it in 3 steps.

In [ ]:
w_b8 = np.ones(6) / 6 # define uniform starting weights.
signs_b8 = np.array([1, 1, 1, 1, 1, -1]) # use +1 for correct and -1 for the one mistake.
alpha_b8 = 0.5 * np.log(5) # use the first-round alpha from epsilon = 1/6.
print("alpha:", round(alpha_b8, 3)) # inspect the update strength.

▶ What you'll see: the same 0.805 vote controls the weight multipliers.

In [ ]:
raw_b8 = w_b8 * np.exp(-alpha_b8 * signs_b8) # compute unnormalized new weights.
new_b8 = raw_b8 / raw_b8.sum() # normalize the weights back to sum 1.
print("raw weights:", np.round(raw_b8, 3)) # inspect values before normalization.
print("normalized:", np.round(new_b8, 3)) # inspect the next-round distribution.
assert np.allclose(np.round(new_b8, 3), np.array([0.1, 0.1, 0.1, 0.1, 0.1, 0.5])) # verify the worked update.

In [ ]:
plt.figure(figsize=(5, 3)) # create a before-after chart.
plt.bar(np.arange(6) - 0.18, w_b8, width=0.36, color="gray", label="before") # draw old weights.
plt.bar(np.arange(6) + 0.18, new_b8, width=0.36, color="teal", label="after") # draw new weights.
plt.title("Basic 8: first AdaBoost reweighting") # title the plot.
plt.xlabel("example index") # label the example axis.
plt.ylabel("weight") # label the weight axis.
plt.legend() # show bar labels.
plt.show() # display the chart.

▶ What you'll see: the mistaken point receives half of the next round's probability mass.

👀 Takeaway: reweighting is AdaBoost's memory of what the ensemble currently struggles with.

### Basic 9 — Add two weak learner votes

**Goal.** Combine two weak rules into one score, because AdaBoost predicts with the sign of a weighted sum. We build it in 3 steps.

In [ ]:
h1_b9 = np.array([-1, -1, 1, 1, 1, 1]) # first weak learner predictions.
h2_b9 = np.array([1, 1, 1, 1, 1, -1]) # second weak learner predictions.
alpha1_b9 = 0.5 * np.log(5) # first learner vote.
alpha2_b9 = 0.5 * np.log(4) # second learner vote from epsilon = 0.2.
print("alphas:", round(alpha1_b9, 3), round(alpha2_b9, 3)) # inspect each vote strength.

▶ What you'll see: the first learner gets a slightly larger vote than the second.

In [ ]:
score_b9 = alpha1_b9 * h1_b9 + alpha2_b9 * h2_b9 # add weighted weak predictions.
pred_b9 = np.sign(score_b9) # convert ensemble scores into class labels.
print("scores:", np.round(score_b9, 3)) # inspect margins before taking signs.
print("predictions:", pred_b9.astype(int)) # inspect ensemble class decisions.

In [ ]:
plt.figure(figsize=(5, 3)) # create a compact ensemble score chart.
plt.bar(np.arange(len(score_b9)), score_b9, color=np.where(score_b9 > 0, "teal", "crimson")) # color bars by predicted class.
plt.axhline(0, color="black", linewidth=1) # mark the decision boundary.
plt.title("Basic 9: two-stump ensemble scores") # title the chart.
plt.xlabel("example index") # label examples.
plt.ylabel("F(x)") # label the ensemble score.
plt.show() # display the chart.

▶ What you'll see: bars above zero are predicted +1 and bars below zero are predicted -1.

👀 Takeaway: AdaBoost is additive in scores, then uses the sign only at the final decision step.

### Basic 10 — Measure margins

**Goal.** Multiply the ensemble score by the true label, because margin summarizes whether a point is correct and how confident the ensemble is. We build it in 3 steps.

In [ ]:
y_b10 = np.array([-1, -1, 1, 1, 1, -1]) # define true labels.
score_b10 = np.array([-0.112, -0.112, 1.498, 1.498, 1.498, -0.112]) # use the rounded two-stump scores.
margin_b10 = y_b10 * score_b10 # compute signed margins.
print("margins:", np.round(margin_b10, 3)) # inspect correctness and confidence together.

▶ What you'll see: every margin is positive, but the negative-class examples have small margins.

In [ ]:
exp_loss_b10 = np.exp(-margin_b10) # compute exponential loss from margins.
print("exponential loss:", np.round(exp_loss_b10, 3)) # inspect which examples remain costly.

In [ ]:
plt.figure(figsize=(5, 3)) # create a compact margin plot.
plt.bar(np.arange(len(margin_b10)), margin_b10, color=np.where(margin_b10 > 0.5, "teal", "orange")) # highlight weak margins.
plt.axhline(0, color="black", linewidth=1) # mark the wrong/correct boundary.
plt.title("Basic 10: ensemble margins") # title the plot.
plt.xlabel("example index") # label examples.
plt.ylabel("yF(x)") # label the margin.
plt.show() # display the chart.

▶ What you'll see: positive margins mean correct predictions, while smaller positive margins are less secure.

👀 Takeaway: AdaBoost keeps pressure on examples with weak margins, not only outright mistakes.

## 🟡 Easy

### Easy 1 — Search for the best weighted stump

**Goal.** Try several thresholds and polarities, because each AdaBoost round needs the weak learner with the lowest current weighted error. We build it in 4 steps.

In [ ]:
x_e1 = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0]) # define the one-feature training set.
y_e1 = np.array([-1, -1, 1, 1, 1, -1]) # define signed labels.
w_e1 = np.ones(6) / 6 # use uniform first-round weights.
thresholds_e1 = np.array([0.75, 1.25, 1.75, 2.25, 2.75]) # candidate split points between examples.
print("thresholds:", thresholds_e1) # inspect the stump search grid.

▶ What you'll see: the weak learner will choose among five possible split locations.

In [ ]:
errors_e1 = [] # store weighted errors for each candidate stump.
preds_e1 = [] # store predictions for each candidate stump.
for threshold_e1 in thresholds_e1: # evaluate each candidate threshold.
    pred_e1 = np.where(x_e1 < threshold_e1, -1, 1) # predict -1 on the left and +1 on the right.
    err_e1 = float(np.sum(w_e1 * (pred_e1 != y_e1))) # compute weighted error for this stump.
    preds_e1.append(pred_e1) # keep the predictions for the winning threshold.
    errors_e1.append(err_e1) # keep the error for plotting.
print("weighted errors:", np.round(errors_e1, 3)) # inspect each candidate's score.

In [ ]:
best_idx_e1 = int(np.argmin(errors_e1)) # choose the threshold with minimum weighted error.
best_threshold_e1 = float(thresholds_e1[best_idx_e1]) # read the winning threshold.
best_error_e1 = float(errors_e1[best_idx_e1]) # read the winning error.
print("best threshold:", best_threshold_e1, "best error:", round(best_error_e1, 3)) # inspect the selected weak learner.
assert best_threshold_e1 == 1.25 and round(best_error_e1, 3) == 0.167 # verify the canonical first stump.

In [ ]:
plt.figure(figsize=(5, 3)) # create a threshold-search plot.
plt.plot(thresholds_e1, errors_e1, marker="o", color="purple") # draw weighted error for each threshold.
plt.axvline(best_threshold_e1, color="red", linestyle="--", label="best") # mark the selected stump.
plt.title("Easy 1: weighted stump search") # title the plot.
plt.xlabel("threshold") # label the split point.
plt.ylabel("weighted error") # label the objective.
plt.legend() # show the best marker.
plt.show() # display the search curve.

▶ What you'll see: threshold 1.25 has the lowest weighted error under uniform weights.

👀 Takeaway: AdaBoost can use any weak-learner training routine as long as it minimizes weighted error.

### Easy 2 — Run two AdaBoost rounds by hand

**Goal.** Execute two full rounds, because the second round should respond to the weights created by the first mistake. We build it in 5 steps.

In [ ]:
x_e2 = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0]) # define feature values.
y_e2 = np.array([-1, -1, 1, 1, 1, -1]) # define labels.
w_e2 = np.ones(6) / 6 # initialize example weights uniformly.
print("start weights:", np.round(w_e2, 3)) # inspect the first-round distribution.

▶ What you'll see: every example begins with equal attention.

In [ ]:
h1_e2 = np.where(x_e2 < 1.25, -1, 1) # choose the first-round stump.
eps1_e2 = float(np.sum(w_e2 * (h1_e2 != y_e2))) # compute its weighted error.
alpha1_e2 = 0.5 * np.log((1 - eps1_e2) / eps1_e2) # compute its vote.
w_e2 = w_e2 * np.exp(-alpha1_e2 * y_e2 * h1_e2) # apply the unnormalized weight update.
w_e2 = w_e2 / w_e2.sum() # normalize weights for the next round.
print("round 1 eps, alpha:", round(eps1_e2, 3), round(alpha1_e2, 3)) # inspect first-round values.
print("after round 1 weights:", np.round(w_e2, 3)) # inspect shifted attention.

In [ ]:
h2_e2 = np.where(x_e2 < 2.75, 1, -1) # choose a second stump that fixes the high-weight last point.
eps2_e2 = float(np.sum(w_e2 * (h2_e2 != y_e2))) # compute weighted error under new weights.
alpha2_e2 = 0.5 * np.log((1 - eps2_e2) / eps2_e2) # compute the second vote.
print("round 2 eps, alpha:", round(eps2_e2, 3), round(alpha2_e2, 3)) # inspect second-round values.
assert round(eps2_e2, 3) == 0.2 # verify the second-round weighted error.

In [ ]:
F_e2 = alpha1_e2 * h1_e2 + alpha2_e2 * h2_e2 # combine the two weak learners.
pred_e2 = np.sign(F_e2) # predict classes by score sign.
acc_e2 = float(np.mean(pred_e2 == y_e2)) # compute training accuracy.
print("F:", np.round(F_e2, 3)) # inspect the ensemble scores.
print("accuracy:", round(acc_e2, 3)) # inspect training fit.
assert round(acc_e2, 3) == 0.833 # verify the two-stump ensemble still misses one fragile point.

In [ ]:
plt.figure(figsize=(5, 3)) # create a score chart.
plt.bar(np.arange(6), F_e2, color=np.where(F_e2 > 0, "teal", "crimson")) # draw ensemble score by example.
plt.axhline(0, color="black", linewidth=1) # show decision boundary.
plt.title("Easy 2: two-round AdaBoost ensemble") # title the chart.
plt.xlabel("example index") # label examples.
plt.ylabel("F(x)") # label ensemble score.
plt.show() # display the chart.

▶ What you'll see: two weak learners fix most examples, but the last point remains a fragile mistake because the first vote is still larger.

👀 Takeaway: boosting changes the next training problem by changing the example distribution.

### Easy 3 — Track empirical score plus cost

**Goal.** Recompute the lesson's raw average and cost-adjusted score, because model selection should use the full decision score rather than raw fit alone. We build it in 4 steps.

In [ ]:
losses_e3 = np.array([0.257, 0.070, 0.522]) # use the three verified per-example losses from the lesson content.
cost_e3 = 0.060 # use the verified method cost.
print("losses:", losses_e3) # inspect the empirical pieces.

▶ What you'll see: the toy training score is built from three inspectable losses.

In [ ]:
raw_e3 = float(np.mean(losses_e3)) # average the empirical losses.
score_e3 = raw_e3 + cost_e3 # add complexity or operational cost.
print("raw average:", round(raw_e3, 3)) # inspect R_S.
print("score with cost:", round(score_e3, 3)) # inspect the selection score.
assert round(raw_e3, 3) == 0.283 and round(score_e3, 3) == 0.343 # verify lesson numbers.

In [ ]:
plt.figure(figsize=(5, 3)) # create a cost-breakdown chart.
plt.bar(["raw R_S", "cost", "total"], [raw_e3, cost_e3, score_e3], color=["gray", "orange", "teal"]) # compare raw and adjusted quantities.
plt.title("Easy 3: fit plus cost") # title the plot.
plt.ylabel("score component") # label the score scale.
plt.show() # display the breakdown.

In [ ]:
print("cost share of total:", round(cost_e3 / score_e3, 3)) # inspect how much of the final score is the guardrail.
assert round(cost_e3 / score_e3, 3) == 0.175 # verify the arithmetic.

▶ What you'll see: the cost is small but large enough to change a close comparison.

👀 Takeaway: the score used for selection is not the same thing as the raw training average.

### Easy 4 — Compare boosted candidates by full score

**Goal.** Compare baseline, flexible, and stabilized candidates, because the lowest full score is the one to carry forward in the verified toy case. We build it in 4 steps.

In [ ]:
baseline_e4 = 0.343 # cost-adjusted baseline score from Easy 3.
flexible_e4 = 0.379 # score for a more flexible alternative.
stable_e4 = 0.80 * baseline_e4 # stabilized score after a 20 percent reduction.
scores_e4 = np.array([baseline_e4, flexible_e4, stable_e4]) # collect candidates for comparison.
print("scores:", np.round(scores_e4, 3)) # inspect the three decision scores.

▶ What you'll see: lower is better, and the stabilized candidate is visibly smallest.

In [ ]:
gap_e4 = flexible_e4 - baseline_e4 # compute the absolute gap between flexible and baseline.
rel_gap_e4 = gap_e4 / flexible_e4 # compute the gap relative to the flexible score.
print("gap:", round(gap_e4, 3), "relative gap:", round(rel_gap_e4, 3)) # inspect evidence size.
assert round(gap_e4, 3) == 0.036 and round(rel_gap_e4, 3) == 0.095 # verify lesson numbers.

In [ ]:
names_e4 = np.array(["baseline", "flexible", "stable"]) # name candidate scores.
winner_e4 = names_e4[int(np.argmin(scores_e4))] # choose the lowest score.
print("winner:", winner_e4) # inspect the selected model.
assert winner_e4 == "stable" # verify the verified toy decision.

In [ ]:
plt.figure(figsize=(5, 3)) # create a model comparison chart.
plt.bar(names_e4, scores_e4, color=["gray", "crimson", "teal"]) # draw each full score.
plt.title("Easy 4: full-score comparison") # title the chart.
plt.ylabel("decision score, lower is better") # label the decision scale.
plt.show() # display the comparison.

▶ What you'll see: the stabilized score 0.274 is the minimum.

👀 Takeaway: close boosting improvements should be judged after cost, uncertainty, and stability are included.

### Easy 5 — Visualize margin improvement across rounds

**Goal.** Track margins after one and two rounds, because AdaBoost's real objective is to push examples toward safer positive margins. We build it in 4 steps.

In [ ]:
y_e5 = np.array([-1, -1, 1, 1, 1, -1]) # define true labels.
h1_e5 = np.array([-1, -1, 1, 1, 1, 1]) # define first stump predictions.
h2_e5 = np.array([1, 1, 1, 1, 1, -1]) # define second stump predictions.
a1_e5 = 0.5 * np.log(5) # first stump vote.
a2_e5 = 0.5 * np.log(4) # second stump vote.
print("votes:", round(a1_e5, 3), round(a2_e5, 3)) # inspect weak-learner weights.

▶ What you'll see: each stump contributes a positive amount to the ensemble score.

In [ ]:
F1_e5 = a1_e5 * h1_e5 # compute one-round ensemble score.
F2_e5 = F1_e5 + a2_e5 * h2_e5 # compute two-round ensemble score.
margin1_e5 = y_e5 * F1_e5 # compute one-round margins.
margin2_e5 = y_e5 * F2_e5 # compute two-round margins.
print("round 1 margins:", np.round(margin1_e5, 3)) # inspect margins before round 2.
print("round 2 margins:", np.round(margin2_e5, 3)) # inspect margins after round 2.

In [ ]:
loss1_e5 = float(np.sum(np.exp(-margin1_e5))) # compute total exponential loss after one round.
loss2_e5 = float(np.sum(np.exp(-margin2_e5))) # compute total exponential loss after two rounds.
print("exp loss round1 -> round2:", round(loss1_e5, 3), "->", round(loss2_e5, 3)) # inspect loss improvement.
assert loss2_e5 < loss1_e5 # verify the second round lowers exponential loss.

In [ ]:
plt.figure(figsize=(5, 3)) # create a margin comparison plot.
plt.plot(margin1_e5, marker="o", label="after round 1") # draw one-round margins.
plt.plot(margin2_e5, marker="s", label="after round 2") # draw two-round margins.
plt.axhline(0, color="black", linewidth=1) # mark wrong/correct margin boundary.
plt.title("Easy 5: margins across boosting rounds") # title the plot.
plt.xlabel("example index") # label examples.
plt.ylabel("margin yF(x)") # label margin scale.
plt.legend() # show curve labels.
plt.show() # display the plot.

▶ What you'll see: the previously wrong example moves from a large negative margin toward zero, even though it is not fully fixed yet.

👀 Takeaway: AdaBoost's additive votes can improve the margin distribution even before every classification is fixed.

## 🔴 Advanced

### Advanced 1 — Build a reusable stump-training loop

**Goal.** Train stumps for several rounds from scratch, because a real AdaBoost implementation repeatedly solves a weighted weak-learning problem. We build it in 5 steps.

In [ ]:
x_a1 = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0]) # define the feature values.
y_a1 = np.array([-1, -1, 1, 1, 1, -1]) # define signed labels.
thresholds_a1 = np.array([0.75, 1.25, 1.75, 2.25, 2.75]) # candidate thresholds.
w_a1 = np.ones(6) / 6 # initialize example weights.
print("initial weights:", np.round(w_a1, 3)) # inspect the starting distribution.

▶ What you'll see: the loop starts from the same uniform distribution as the basics.

In [ ]:
alphas_a1 = [] # store learner votes.
stumps_a1 = [] # store each stump as threshold and polarity.
weight_history_a1 = [w_a1.copy()] # store weights for visualization.
for round_a1 in range(3): # train three boosting rounds.
    best_err_a1 = 1.0 # initialize the best weighted error.
    best_pred_a1 = None # initialize the best prediction vector.
    best_rule_a1 = None # initialize the best stump description.
    for threshold_a1 in thresholds_a1: # search threshold values.
        for polarity_a1 in [-1, 1]: # allow either orientation of the stump.
            pred_a1 = np.where(x_a1 < threshold_a1, -1, 1) * polarity_a1 # apply threshold and polarity.
            err_a1 = float(np.sum(w_a1 * (pred_a1 != y_a1))) # compute weighted error.
            if err_a1 > 0.5: # flip overly bad learners so error is at most 0.5.
                err_a1 = 1 - err_a1 # flipped weighted error.
                pred_a1 = -pred_a1 # flipped predictions.
                polarity_a1 = -polarity_a1 # record flipped orientation.
            if err_a1 < best_err_a1: # keep the best stump found so far.
                best_err_a1 = err_a1 # update best error.
                best_pred_a1 = pred_a1.copy() # update best predictions.
                best_rule_a1 = (float(threshold_a1), int(polarity_a1)) # update best rule.
    alpha_a1 = 0.5 * np.log((1 - best_err_a1) / max(best_err_a1, 1e-12)) # compute vote with a safety floor.
    w_a1 = w_a1 * np.exp(-alpha_a1 * y_a1 * best_pred_a1) # update example weights.
    w_a1 = w_a1 / w_a1.sum() # normalize the next-round distribution.
    alphas_a1.append(alpha_a1) # store vote.
    stumps_a1.append(best_rule_a1) # store stump rule.
    weight_history_a1.append(w_a1.copy()) # store weights after this round.
print("stumps:", stumps_a1) # inspect learned weak rules.
print("alphas:", np.round(alphas_a1, 3)) # inspect votes.

In [ ]:
F_a1 = np.zeros_like(y_a1, dtype=float) # initialize additive ensemble scores.
for alpha_a1, (threshold_a1, polarity_a1) in zip(alphas_a1, stumps_a1): # loop over learned stumps.
    F_a1 += alpha_a1 * (np.where(x_a1 < threshold_a1, -1, 1) * polarity_a1) # add this weak learner's vote.
pred_a1 = np.sign(F_a1) # convert scores to class predictions.
print("final scores:", np.round(F_a1, 3)) # inspect additive scores.
print("training accuracy:", round(float(np.mean(pred_a1 == y_a1)), 3)) # inspect final fit.
assert float(np.mean(pred_a1 == y_a1)) >= 0.833 # verify the ensemble learned useful structure.

In [ ]:
W_a1 = np.vstack(weight_history_a1) # stack weight distributions by round.
print("weight history shape:", W_a1.shape) # inspect rows as rounds and columns as examples.

In [ ]:
plt.figure(figsize=(6, 3)) # create a heatmap of example attention over rounds.
plt.imshow(W_a1, cmap="viridis", aspect="auto") # draw weights as colors.
plt.colorbar(label="weight") # add a color scale.
plt.title("Advanced 1: weights across rounds") # title the heatmap.
plt.xlabel("example index") # label examples.
plt.ylabel("round") # label boosting round.
plt.show() # display the heatmap.

▶ What you'll see: mass moves toward the examples that remain difficult under the current ensemble.

👀 Takeaway: the weak learner is retrained on a changing distribution, so each round solves a different weighted problem.

### Advanced 2 — Compare AdaBoost with a single stump on validation data

**Goal.** Hold out a few points from a synthetic line and compare validation error, because boosting should be judged on future-like data rather than training fit alone. We build it in 5 steps.

In [ ]:
x_a2 = np.linspace(0, 6, 18) # create a small one-dimensional dataset.
y_a2 = np.where((x_a2 > 1.4) & (x_a2 < 4.8), 1, -1) # define an interval-shaped positive class.
y_a2[[4, 13]] *= -1 # add two label irregularities to make the task imperfect.
train_a2 = np.arange(0, 18, 2) # use even-indexed points for training.
val_a2 = np.arange(1, 18, 2) # use odd-indexed points for validation.
print("train size:", len(train_a2), "validation size:", len(val_a2)) # inspect the split.

▶ What you'll see: the model will learn from half the points and be checked on the other half.

In [ ]:
xt_a2 = x_a2[train_a2] # select training features.
yt_a2 = y_a2[train_a2] # select training labels.
thresholds_a2 = (xt_a2[:-1] + xt_a2[1:]) / 2 # define candidate thresholds between training points.
w_a2 = np.ones_like(yt_a2, dtype=float) / len(yt_a2) # initialize training weights.
print("candidate thresholds:", np.round(thresholds_a2, 2)) # inspect weak-learner candidates.

In [ ]:
alphas_a2 = [] # store boosting votes.
rules_a2 = [] # store boosting rules.
for round_a2 in range(4): # train four boosting rounds.
    best_err_a2 = 1.0 # initialize best error.
    best_rule_a2 = None # initialize best rule.
    best_pred_a2 = None # initialize best predictions.
    for thr_a2 in thresholds_a2: # search thresholds.
        for left_a2 in [-1, 1]: # search left-side labels.
            pred_a2 = np.where(xt_a2 < thr_a2, left_a2, -left_a2) # make stump predictions.
            err_a2 = float(np.sum(w_a2 * (pred_a2 != yt_a2))) # compute weighted error.
            if err_a2 < best_err_a2: # retain the best stump.
                best_err_a2 = err_a2 # update best error.
                best_rule_a2 = (float(thr_a2), int(left_a2)) # update rule.
                best_pred_a2 = pred_a2.copy() # update predictions.
    alpha_a2 = 0.5 * np.log((1 - best_err_a2) / max(best_err_a2, 1e-12)) # compute learner vote.
    w_a2 = w_a2 * np.exp(-alpha_a2 * yt_a2 * best_pred_a2) # update weights.
    w_a2 = w_a2 / w_a2.sum() # normalize weights.
    alphas_a2.append(alpha_a2) # store vote.
    rules_a2.append(best_rule_a2) # store rule.
print("rules:", rules_a2) # inspect selected stumps.
print("alphas:", np.round(alphas_a2, 3)) # inspect votes.

In [ ]:
def predict_rules_a2(x_values_a2, rules_in_a2, alphas_in_a2): # define a local prediction helper for this example.
    score_out_a2 = np.zeros_like(x_values_a2, dtype=float) # initialize scores.
    for alpha_in_a2, (thr_in_a2, left_in_a2) in zip(alphas_in_a2, rules_in_a2): # add each stump.
        score_out_a2 += alpha_in_a2 * np.where(x_values_a2 < thr_in_a2, left_in_a2, -left_in_a2) # accumulate weighted predictions.
    return np.sign(score_out_a2), score_out_a2 # return labels and raw scores.
boost_pred_val_a2, boost_score_val_a2 = predict_rules_a2(x_a2[val_a2], rules_a2, alphas_a2) # predict validation labels with boosting.
single_pred_val_a2 = np.where(x_a2[val_a2] < rules_a2[0][0], rules_a2[0][1], -rules_a2[0][1]) # predict validation labels with only the first stump.
boost_err_a2 = float(np.mean(boost_pred_val_a2 != y_a2[val_a2])) # compute boosted validation error.
single_err_a2 = float(np.mean(single_pred_val_a2 != y_a2[val_a2])) # compute single-stump validation error.
print("single-stump val error:", round(single_err_a2, 3), "boosted val error:", round(boost_err_a2, 3)) # inspect validation comparison.

In [ ]:
plt.figure(figsize=(6, 3)) # create a validation visualization.
plt.scatter(x_a2[train_a2], y_a2[train_a2], c="gray", label="train", s=60) # draw training points.
plt.scatter(x_a2[val_a2], y_a2[val_a2], c=np.where(boost_pred_val_a2 == y_a2[val_a2], "teal", "crimson"), label="validation", s=90) # color validation correctness.
plt.plot(x_a2[val_a2], boost_score_val_a2 / max(1, np.max(np.abs(boost_score_val_a2))), color="purple", label="scaled boosted score") # plot scaled scores for inspection.
plt.yticks([-1, 1]); plt.title("Advanced 2: validation check") # title the plot.
plt.xlabel("x") # label feature axis.
plt.legend() # show legend.
plt.show() # display the plot.

▶ What you'll see: validation points reveal whether the boosted boundary improves on the first stump outside training.

👀 Takeaway: AdaBoost's attractive training behavior must still survive a held-out comparison.

### Advanced 3 — Sweep the number of boosting rounds

**Goal.** Compare train and validation error as rounds increase, because too few rounds underfit and too many can chase idiosyncrasies. We build it in 5 steps.

In [ ]:
x_a3 = np.linspace(0, 6, 24) # create a slightly larger one-dimensional dataset.
y_a3 = np.where(np.sin(1.3 * x_a3) > 0, 1, -1) # define a nonlinear signed pattern.
y_a3[[3, 17]] *= -1 # add two irregular labels.
train_a3 = np.arange(0, 24, 2) # choose even points for training.
val_a3 = np.arange(1, 24, 2) # choose odd points for validation.
round_grid_a3 = np.array([1, 2, 4, 8]) # choose boosting depths to compare.
print("round grid:", round_grid_a3) # inspect candidate ensemble sizes.

▶ What you'll see: the sweep will test ensembles from one stump to eight stumps.

In [ ]:
def train_boost_a3(x_train_a3, y_train_a3, rounds_a3): # define a local AdaBoost trainer for the sweep.
    thresholds_local_a3 = (x_train_a3[:-1] + x_train_a3[1:]) / 2 # candidate split points.
    weights_local_a3 = np.ones_like(y_train_a3, dtype=float) / len(y_train_a3) # uniform starting weights.
    rules_local_a3 = [] # store rules.
    alphas_local_a3 = [] # store votes.
    for _ in range(rounds_a3): # train requested number of rounds.
        best_error_local_a3 = 1.0 # initialize best error.
        for thr_local_a3 in thresholds_local_a3: # search thresholds.
            for left_local_a3 in [-1, 1]: # search polarities.
                pred_local_a3 = np.where(x_train_a3 < thr_local_a3, left_local_a3, -left_local_a3) # stump predictions.
                err_local_a3 = float(np.sum(weights_local_a3 * (pred_local_a3 != y_train_a3))) # weighted error.
                if err_local_a3 < best_error_local_a3: # keep best stump.
                    best_error_local_a3 = err_local_a3 # update best error.
                    best_pred_local_a3 = pred_local_a3.copy() # update best predictions.
                    best_rule_local_a3 = (float(thr_local_a3), int(left_local_a3)) # update best rule.
        alpha_local_a3 = 0.5 * np.log((1 - best_error_local_a3) / max(best_error_local_a3, 1e-12)) # compute vote.
        weights_local_a3 *= np.exp(-alpha_local_a3 * y_train_a3 * best_pred_local_a3) # update weights.
        weights_local_a3 /= weights_local_a3.sum() # normalize weights.
        rules_local_a3.append(best_rule_local_a3) # store rule.
        alphas_local_a3.append(alpha_local_a3) # store vote.
    return rules_local_a3, alphas_local_a3 # return learned ensemble.
print("trainer ready") # confirm the helper is defined.

In [ ]:
def predict_boost_a3(x_values_a3, rules_in_a3, alphas_in_a3): # define a local predictor for the sweep.
    scores_out_a3 = np.zeros_like(x_values_a3, dtype=float) # initialize additive scores.
    for alpha_in_a3, (thr_in_a3, left_in_a3) in zip(alphas_in_a3, rules_in_a3): # loop over weak learners.
        scores_out_a3 += alpha_in_a3 * np.where(x_values_a3 < thr_in_a3, left_in_a3, -left_in_a3) # add weighted stump output.
    return np.sign(scores_out_a3) # return signed predictions.
train_errs_a3 = [] # store training errors.
val_errs_a3 = [] # store validation errors.
for rounds_a3 in round_grid_a3: # evaluate each number of rounds.
    rules_a3, alphas_a3 = train_boost_a3(x_a3[train_a3], y_a3[train_a3], int(rounds_a3)) # train an ensemble.
    pred_train_a3 = predict_boost_a3(x_a3[train_a3], rules_a3, alphas_a3) # predict training set.
    pred_val_a3 = predict_boost_a3(x_a3[val_a3], rules_a3, alphas_a3) # predict validation set.
    train_errs_a3.append(float(np.mean(pred_train_a3 != y_a3[train_a3]))) # store training error.
    val_errs_a3.append(float(np.mean(pred_val_a3 != y_a3[val_a3]))) # store validation error.
print("train errors:", np.round(train_errs_a3, 3)) # inspect fit by ensemble size.
print("validation errors:", np.round(val_errs_a3, 3)) # inspect future-like performance.

In [ ]:
best_rounds_a3 = int(round_grid_a3[int(np.argmin(val_errs_a3))]) # choose the validation-best ensemble size.
print("best rounds by validation:", best_rounds_a3) # inspect selected round count.

In [ ]:
plt.figure(figsize=(5, 3)) # create the round-sweep plot.
plt.plot(round_grid_a3, train_errs_a3, marker="o", label="train error") # draw training error.
plt.plot(round_grid_a3, val_errs_a3, marker="s", label="validation error") # draw validation error.
plt.axvline(best_rounds_a3, color="red", linestyle="--", label="best validation") # mark best round count.
plt.title("Advanced 3: rounds are a flexibility knob") # title the plot.
plt.xlabel("boosting rounds") # label x axis.
plt.ylabel("classification error") # label y axis.
plt.legend() # show labels.
plt.show() # display the sweep.

▶ What you'll see: training error tends to improve with more rounds, while validation error decides which improvement is useful.

👀 Takeaway: the number of boosting rounds is a capacity knob and should be selected with validation data.

### Advanced 4 — Inspect the exponential-loss bound

**Goal.** Track zero-one error and exponential loss together, because AdaBoost optimizes a smooth upper bound rather than direct classification error. We build it in 4 steps.

In [ ]:
y_a4 = np.array([-1, -1, 1, 1, 1, -1]) # define labels.
learners_a4 = np.array([[-1, -1, 1, 1, 1, 1], [1, 1, 1, 1, 1, -1], [-1, 1, 1, 1, -1, -1]]) # define three weak prediction vectors.
alphas_a4 = np.array([0.805, 0.693, 0.255]) # define learner votes from plausible weighted errors.
print("learners shape:", learners_a4.shape) # inspect number of learners and examples.

▶ What you'll see: three rows of weak predictions are ready to be accumulated.

In [ ]:
scores_a4 = np.zeros_like(y_a4, dtype=float) # initialize ensemble scores.
zero_one_a4 = [] # store classification error after each round.
exp_bound_a4 = [] # store average exponential loss after each round.
for t_a4 in range(len(alphas_a4)): # accumulate learners one at a time.
    scores_a4 += alphas_a4[t_a4] * learners_a4[t_a4] # add the current weak learner's vote.
    margins_a4 = y_a4 * scores_a4 # compute current margins.
    zero_one_a4.append(float(np.mean(np.sign(scores_a4) != y_a4))) # compute zero-one training error.
    exp_bound_a4.append(float(np.mean(np.exp(-margins_a4)))) # compute average exponential loss.
print("zero-one error:", np.round(zero_one_a4, 3)) # inspect discrete error.
print("exponential loss:", np.round(exp_bound_a4, 3)) # inspect smooth bound.

In [ ]:
assert exp_bound_a4[-1] >= zero_one_a4[-1] # verify exponential loss upper-bounds zero-one error on average for these margins.
print("final margin min/max:", round(float(np.min(margins_a4)), 3), round(float(np.max(margins_a4)), 3)) # inspect margin range.

In [ ]:
plt.figure(figsize=(5, 3)) # create a bound-tracking plot.
plt.plot(np.arange(1, 4), zero_one_a4, marker="o", label="zero-one error") # draw classification error.
plt.plot(np.arange(1, 4), exp_bound_a4, marker="s", label="exponential loss") # draw exponential loss.
plt.title("Advanced 4: AdaBoost follows a smooth bound") # title the plot.
plt.xlabel("round") # label boosting round.
plt.ylabel("average loss") # label loss scale.
plt.legend() # show curve labels.
plt.show() # display the comparison.

▶ What you'll see: zero-one error can flatten while exponential loss still changes with margin quality.

👀 Takeaway: AdaBoost's update is driven by exponential-loss pressure on margins, not by directly minimizing the stepwise error count.

### Advanced 5 — Show sensitivity to mislabeled outliers

**Goal.** Add a deliberately flipped point and watch its weight grow, because AdaBoost can over-focus on noisy examples if the weak learner cannot resolve them. We build it in 5 steps.

In [ ]:
x_a5 = np.linspace(0, 5, 11) # create ordered feature values.
y_clean_a5 = np.where(x_a5 < 2.5, -1, 1) # define a clean threshold pattern.
y_noisy_a5 = y_clean_a5.copy() # copy labels before adding noise.
y_noisy_a5[2] = 1 # flip one left-side point into a mislabeled positive outlier.
print("noisy labels:", y_noisy_a5) # inspect the label flip.

▶ What you'll see: one point disagrees with the otherwise clean left/right pattern.

In [ ]:
thresholds_a5 = (x_a5[:-1] + x_a5[1:]) / 2 # candidate stump thresholds.
w_a5 = np.ones_like(x_a5) / len(x_a5) # initialize weights uniformly.
outlier_idx_a5 = 2 # remember the noisy point index.
outlier_weights_a5 = [w_a5[outlier_idx_a5]] # store its weight over rounds.
train_errors_a5 = [] # store training error over rounds.
print("outlier x:", x_a5[outlier_idx_a5], "label:", y_noisy_a5[outlier_idx_a5]) # inspect the noisy example.

In [ ]:
score_a5 = np.zeros_like(x_a5, dtype=float) # initialize ensemble scores.
for round_a5 in range(6): # run several boosting rounds.
    best_err_a5 = 1.0 # initialize best weighted error.
    for thr_a5 in thresholds_a5: # search thresholds.
        for left_a5 in [-1, 1]: # search polarities.
            pred_a5 = np.where(x_a5 < thr_a5, left_a5, -left_a5) # stump predictions.
            err_a5 = float(np.sum(w_a5 * (pred_a5 != y_noisy_a5))) # weighted error.
            if err_a5 < best_err_a5: # keep the best stump.
                best_err_a5 = err_a5 # update best error.
                best_pred_a5 = pred_a5.copy() # update best predictions.
    alpha_a5 = 0.5 * np.log((1 - best_err_a5) / max(best_err_a5, 1e-12)) # compute vote.
    score_a5 += alpha_a5 * best_pred_a5 # update ensemble score.
    w_a5 *= np.exp(-alpha_a5 * y_noisy_a5 * best_pred_a5) # update weights.
    w_a5 /= w_a5.sum() # normalize weights.
    outlier_weights_a5.append(float(w_a5[outlier_idx_a5])) # record outlier weight.
    train_errors_a5.append(float(np.mean(np.sign(score_a5) != y_noisy_a5))) # record training error.
print("outlier weights:", np.round(outlier_weights_a5, 3)) # inspect attention on the noisy point.
print("training errors:", np.round(train_errors_a5, 3)) # inspect fit over rounds.

In [ ]:
assert max(outlier_weights_a5) > outlier_weights_a5[0] # verify the noisy example received more attention at some point.
print("max outlier weight:", round(float(max(outlier_weights_a5)), 3)) # inspect the peak attention.

In [ ]:
plt.figure(figsize=(5, 3)) # create an outlier-attention plot.
plt.plot(outlier_weights_a5, marker="o", color="crimson") # draw noisy-point weight over rounds.
plt.title("Advanced 5: noisy point can attract weight") # title the plot.
plt.xlabel("round") # label round number.
plt.ylabel("outlier weight") # label the attention mass.
plt.show() # display the curve.

In [ ]:
plt.figure(figsize=(5, 3)) # create a final data plot.
plt.scatter(x_a5, y_noisy_a5, s=80 + 500 * w_a5, c=np.where(y_noisy_a5 > 0, "teal", "crimson")) # size points by final AdaBoost weight.
plt.title("Advanced 5: final weights on noisy data") # title the scatter.
plt.xlabel("x") # label feature axis.
plt.ylabel("label") # label class axis.
plt.yticks([-1, 1]) # show signed labels.
plt.show() # display weighted points.

▶ What you'll see: the flipped point can become visually large because repeated mistakes keep increasing its weight.

👀 Takeaway: AdaBoost is powerful, but label noise and outliers can consume attention unless validation, early stopping, or robust variants control the focus.